In [17]:
!pip install fastapi uvicorn pyngrok nest-asyncio python-multipart pdfplumber python-docx pandas scikit-learn jinja2 aiofiles

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import HTMLResponse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pdfplumber
import docx
import pandas as pd
import re
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import os

# ==============================
# FASTAPI APP
# ==============================

app = FastAPI()

# ==============================
# REQUIRED SKILLS
# ==============================

required_skills = [
    "python",
    "sql",
    "fastapi",
    "machine learning",
    "pandas",
    "numpy",
    "git",
    "data analysis",
    "api"
]

# ==============================
# EXTRACT TEXT FROM PDF
# ==============================

def extract_pdf_text(file_path):
    text = ""

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            extracted = page.extract_text()

            if extracted:
                text += extracted + " "

    return text

# ==============================
# EXTRACT TEXT FROM DOCX
# ==============================

def extract_docx_text(file_path):
    doc = docx.Document(file_path)

    text = ""

    for para in doc.paragraphs:
        text += para.text + " "

    return text

# ==============================
# CLEAN TEXT
# ==============================

def clean_text(text):
    text = text.lower()

    text = re.sub(r"[^a-zA-Z0-9 ]", "", text)

    return text

# ==============================
# EXTRACT MATCHED SKILLS
# ==============================

def extract_skills(text):
    found_skills = []

    for skill in required_skills:
        if skill.lower() in text.lower():
            found_skills.append(skill)

    return found_skills

# ==============================
# ATS SCORE CALCULATION
# ==============================

def calculate_score(resume_text, jd_text):

    documents = [resume_text, jd_text]

    tfidf = TfidfVectorizer()

    tfidf_matrix = tfidf.fit_transform(documents)

    similarity = cosine_similarity(
        tfidf_matrix[0:1],
        tfidf_matrix[1:2]
    )

    score = round(similarity[0][0] * 100, 2)

    return score

# ==============================
# HOME PAGE
# ==============================

@app.get("/", response_class=HTMLResponse)
async def home():

    return f"""

    <!DOCTYPE html>
    <html>

    <head>

        <title>ATS Resume Screening Tool</title>

        <script src="https://cdn.tailwindcss.com"></script>

        <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>

    </head>

    <body class="bg-slate-950 text-white min-h-screen">

        <div class="max-w-7xl mx-auto p-8">

            <!-- HEADER -->

            <div class="flex justify-between items-center mb-10">

                <div>
                    <h1 class="text-5xl font-bold bg-gradient-to-r from-cyan-400 to-blue-500 text-transparent bg-clip-text">
                        ATS Resume Intelligence Dashboard
                    </h1>

                    <p class="text-slate-400 mt-3 text-lg">
                        AI Powered Automated Resume Screening System
                    </p>
                </div>

                <div class="bg-emerald-500/20 border border-emerald-500/30 text-emerald-400 px-6 py-3 rounded-2xl">
                    ● FastAPI Connected
                </div>

            </div>

            <!-- STATS -->

            <div class="grid grid-cols-1 md:grid-cols-4 gap-6 mb-10">

                <div class="bg-slate-900 p-6 rounded-3xl border border-slate-800">
                    <p class="text-slate-400">Total Resumes</p>
                    <h2 class="text-4xl font-bold mt-4 text-cyan-400">124</h2>
                </div>

                <div class="bg-slate-900 p-6 rounded-3xl border border-slate-800">
                    <p class="text-slate-400">Shortlisted</p>
                    <h2 class="text-4xl font-bold mt-4 text-emerald-400">37</h2>
                </div>

                <div class="bg-slate-900 p-6 rounded-3xl border border-slate-800">
                    <p class="text-slate-400">Rejected</p>
                    <h2 class="text-4xl font-bold mt-4 text-red-400">87</h2>
                </div>

                <div class="bg-slate-900 p-6 rounded-3xl border border-slate-800">
                    <p class="text-slate-400">Average ATS Score</p>
                    <h2 class="text-4xl font-bold mt-4 text-yellow-400">74%</h2>
                </div>

            </div>

            <!-- MAIN GRID -->

            <div class="grid grid-cols-1 lg:grid-cols-3 gap-8">

                <!-- LEFT PANEL -->

                <div class="space-y-8">

                    <!-- UPLOAD CARD -->

                    <div class="bg-slate-900 p-8 rounded-3xl border border-slate-800">

                        <h2 class="text-2xl font-semibold mb-6">
                            Upload Resume
                        </h2>

                        <form action="/analyze" method="post" enctype="multipart/form-data">

                            <div class="border-2 border-dashed border-slate-700 rounded-3xl p-10 text-center bg-slate-950">

                                <div class="text-6xl mb-4">📄</div>

                                <p class="text-xl font-semibold">
                                    Drag & Drop Resume
                                </p>

                                <p class="text-slate-400 mt-2 mb-6">
                                    PDF or DOCX files supported
                                </p>

                                <input
                                    type="file"
                                    name="resume"
                                    class="w-full bg-slate-800 rounded-xl p-3 mb-6"
                                    required
                                >

                                <textarea
                                    name="job_description"
                                    placeholder="Paste Job Description Here..."
                                    class="w-full bg-slate-800 rounded-2xl p-4 h-40 mb-6"
                                    required
                                ></textarea>

                                <button
                                    type="submit"
                                    class="w-full bg-cyan-500 hover:bg-cyan-600 text-black font-bold py-4 rounded-2xl transition-all duration-300"
                                >
                                    Analyze Resume
                                </button>

                            </div>

                        </form>

                    </div>

                    <!-- REQUIRED SKILLS -->

                    <div class="bg-slate-900 p-8 rounded-3xl border border-slate-800">

                        <h2 class="text-2xl font-semibold mb-6">
                            Required Skills
                        </h2>

                        <div class="flex flex-wrap gap-3">

                            {''.join([
                                f'<span class="bg-cyan-500/10 text-cyan-400 border border-cyan-500/30 px-4 py-2 rounded-full">{skill}</span>'
                                for skill in required_skills
                            ])}

                        </div>

                    </div>

                </div>

                <!-- RIGHT PANEL -->

                <div class="lg:col-span-2 space-y-8">

                    <!-- CHARTS -->

                    <div class="grid grid-cols-1 lg:grid-cols-2 gap-8">

                        <div class="bg-slate-900 p-8 rounded-3xl border border-slate-800">

                            <h2 class="text-2xl font-semibold mb-6">
                                Candidate Ranking Scores
                            </h2>

                            <canvas id="barChart"></canvas>

                        </div>

                        <div class="bg-slate-900 p-8 rounded-3xl border border-slate-800">

                            <h2 class="text-2xl font-semibold mb-6">
                                Hiring Distribution
                            </h2>

                            <canvas id="pieChart"></canvas>

                        </div>

                    </div>

                    <!-- TABLE -->

                    <div class="bg-slate-900 p-8 rounded-3xl border border-slate-800">

                        <h2 class="text-2xl font-semibold mb-6">
                            Top Ranked Candidates
                        </h2>

                        <table class="w-full">

                            <thead>

                                <tr class="border-b border-slate-700 text-slate-400">

                                    <th class="text-left pb-4">Candidate</th>
                                    <th class="text-left pb-4">ATS Score</th>
                                    <th class="text-left pb-4">Status</th>

                                </tr>

                            </thead>

                            <tbody>

                                <tr class="border-b border-slate-800">
                                    <td class="py-4">John Doe</td>
                                    <td class="py-4 text-cyan-400">82%</td>
                                    <td class="py-4 text-emerald-400">Shortlisted</td>
                                </tr>

                                <tr class="border-b border-slate-800">
                                    <td class="py-4">Emily Smith</td>
                                    <td class="py-4 text-cyan-400">74%</td>
                                    <td class="py-4 text-emerald-400">Shortlisted</td>
                                </tr>

                                <tr>
                                    <td class="py-4">Alex Johnson</td>
                                    <td class="py-4 text-cyan-400">39%</td>
                                    <td class="py-4 text-red-400">Rejected</td>
                                </tr>

                            </tbody>

                        </table>

                    </div>

                </div>

            </div>

        </div>

        <script>

            const barCtx = document.getElementById('barChart');

            new Chart(barCtx, {{

                type: 'bar',

                data: {{
                    labels: ['John', 'Emily', 'Alex', 'Sophia'],
                    datasets: [{{
                        label: 'ATS Score',
                        data: [82, 74, 39, 91],
                        backgroundColor: '#06b6d4'
                    }}]
                }}

            }});

            const pieCtx = document.getElementById('pieChart');

            new Chart(pieCtx, {{

                type: 'pie',

                data: {{
                    labels: ['Shortlisted', 'Rejected'],
                    datasets: [{{
                        data: [37, 87],
                        backgroundColor: ['#10b981', '#ef4444']
                    }}]
                }}

            }});

        </script>

    </body>

    </html>

    """

# ==============================
# ANALYZE RESUME
# ==============================

@app.post("/analyze", response_class=HTMLResponse)
async def analyze_resume(
    resume: UploadFile = File(...),
    job_description: str = Form(...)
):

    file_location = f"./{resume.filename}"

    with open(file_location, "wb") as file:
        file.write(await resume.read())

    # PDF
    if resume.filename.endswith(".pdf"):
        resume_text = extract_pdf_text(file_location)

    # DOCX
    elif resume.filename.endswith(".docx"):
        resume_text = extract_docx_text(file_location)

    else:
        return "Unsupported File Format"

    resume_text = clean_text(resume_text)

    jd_text = clean_text(job_description)

    matched_skills = extract_skills(resume_text)

    score = calculate_score(resume_text, jd_text)

    status = "Shortlisted" if score >= 60 else "Rejected"

    color = "emerald" if score >= 60 else "red"

    # SAVE REPORT

    report = pd.DataFrame({
        "Resume": [resume.filename],
        "ATS Score": [score],
        "Status": [status],
        "Matched Skills": [", ".join(matched_skills)]
    })

    report.to_csv("ats_report.csv", index=False)

    return f"""

    <!DOCTYPE html>

    <html>

    <head>

        <title>ATS Result</title>

        <script src="https://cdn.tailwindcss.com"></script>

    </head>

    <body class="bg-slate-950 text-white min-h-screen flex items-center justify-center p-10">

        <div class="bg-slate-900 border border-slate-800 rounded-3xl p-10 w-full max-w-4xl">

            <h1 class="text-4xl font-bold mb-8 text-center">
                Resume Screening Result
            </h1>

            <div class="grid grid-cols-1 md:grid-cols-2 gap-8 mb-10">

                <div class="bg-slate-950 p-6 rounded-2xl">
                    <p class="text-slate-400">Resume File</p>
                    <h2 class="text-2xl font-bold mt-2">{resume.filename}</h2>
                </div>

                <div class="bg-slate-950 p-6 rounded-2xl">
                    <p class="text-slate-400">ATS Match Score</p>
                    <h2 class="text-5xl font-bold text-cyan-400 mt-2">
                        {score}%
                    </h2>
                </div>

            </div>

            <div class="mb-8">

                <h2 class="text-2xl font-semibold mb-4">
                    Matched Skills
                </h2>

                <div class="flex flex-wrap gap-3">

                    {''.join([
                        f'<span class="bg-emerald-500/10 text-emerald-400 border border-emerald-500/30 px-4 py-2 rounded-full">{skill}</span>'
                        for skill in matched_skills
                    ])}

                </div>

            </div>

            <div class="mb-8">

                <h2 class="text-2xl font-semibold mb-4">
                    Final Decision
                </h2>

                <div class="bg-{color}-500/10 border border-{color}-500/30 text-{color}-400 px-6 py-4 rounded-2xl inline-block text-xl font-bold">
                    {status}
                </div>

            </div>

            <div class="bg-slate-950 p-6 rounded-2xl">

                <h2 class="text-2xl font-semibold mb-4">
                    AI Feedback
                </h2>

                <ul class="space-y-4 text-slate-300">

                    <li>✅ Resume successfully parsed</li>
                    <li>✅ ATS keywords identified</li>
                    <li>✅ Resume formatting is ATS friendly</li>
                    <li>⚠ Add more cloud and deployment skills</li>

                </ul>

            </div>

        </div>

    </body>

    </html>

    """

# ==============================
# START SERVER
# ==============================
# ==============================
# START SERVER
# ==============================

import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

# Replace with your real token
ngrok.set_auth_token("3CZ0VJXuTUKg1Ljsr905XBgOr0R_6GEV8fR54XYShAxEcFDTo")

public_url = ngrok.connect(8000)

print("Public URL:", public_url)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000
)

server = uvicorn.Server(config)

await server.serve()

nest_asyncio.apply()

# ADD YOUR REAL NGROK TOKEN HERE
ngrok.set_auth_token("3CZ0VJXuTUKg1Ljsr905XBgOr0R_6GEV8fR54XYShAxEcFDTo")

public_url = ngrok.connect(8000)

print("Public URL:", public_url)

uvicorn.run(app, host="0.0.0.0", port=8000)

Public URL: NgrokTunnel: "https://haziness-phoniness-playoff.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [2831]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     45.251.233.11:0 - "GET / HTTP/1.1" 200 OK
INFO:     45.251.233.11:0 - "POST /analyze HTTP/1.1" 200 OK
